# Does observed replenishment distinguish reversal from continuation?

**MVP 2 · B5–B8; F1.** Compare two explanations of the same liquidity shock.
Owner: researcher. Design refreshed 2026-09-23; awaiting empirical evidence.

Share sample, horizons, costs and risk budgets with [liquidity-taking](liquidity-taking.ipynb).
Retain the event definition and [study protocol](study-protocol.md) before inspecting outcomes.

## 1. Mechanism and competing explanation

A loss of displayed liquidity may be a temporary dislocation followed by replenishment,
or persistent pressure followed by further repricing. Ask whether information already
observed after the shock distinguishes them.

Define depletion relative to past depth and volatility, with price-distance bands that
remain meaningful when the best quote moves. Separate removal of depth from price moving
outside the measurement band. L2 alone cannot distinguish executions from cancellations;
add public trades to test that interpretation rather than assigning causes from quantity changes.

B6 asks whether persistence improves the distinction; B7 controls for recent returns;
B8 asks whether the useful outcome is avoiding fragile liquidity rather than forecasting direction.

## 2. Make event timing causal

1. Detect a shock using only messages received so far, with a training-defined threshold.
2. Start a bounded observation window; define replenishment or persistent depletion in advance.
3. Declare the state only when its condition becomes observable. Keep timeouts and no-replenishment events.
4. Decide after that time; enter after processing and submission delay.
5. Label from that feasible entry under fixed exit rules, not from the earlier shock onset.

Never define a shock using a future local extremum, classify it from eventual recovery, or
enter at the price preceding an observed replenishment. Track shock receipt, condition
availability, decision, entry and exit separately. For a diagnostic onset-to-outcome plot,
mark the non-tradeable pre-decision portion explicitly.

Cluster related shocks into episodes and freeze a cooldown/overlap rule. Exclude sequence
gaps and resynchronization bursts. Report excluded events instead of quietly deleting them.

## 3. Compare matched policies

At each common eligible decision, evaluate reversal, continuation, a recent-return baseline
and abstention under the same delay, size and risk budget. Then test the **conditional**
policy: reverse only after the specified observed replenishment, continue only after
persistent depletion, otherwise abstain. Do not select direction from realized profit.

Condition on spread, volatility, shock side and prior return using training-defined bins.
Add trade flow as a separate ablation. Fix a small horizon grid; split chronologically and
keep overlapping episodes out of opposite partitions.

Specify which sides the product/account can execute. If short exposure is unsupported,
restrict the experiment to admissible policies and events; do not invent a spot short.
An event study with hypothetical independent trades still needs an inventory/overlap-aware
portfolio evaluation before it can establish strategy economics.

## 4. Inspect the event-study output

Populate one instrument/currency, horizon, size and delay scenario per table. All four policy
outcomes must be evaluated on the same events. `condition_available_at` is the time the
predeclared replenishment/persistence or timeout observation is known. Entry/exit are common
counterfactual execution times for this paired event diagnostic. The conditional policy may
abstain; its zero outcome remains in the row. A state is not assigned from future recovery.

`episode_id` groups dependent shocks. `*_net_quote` includes complete entry, exit and costs.
`evidence_ref` points to the retained event and execution trace. The worksheet checks timing
and summarizes supplied outcomes; it does not reconstruct events or manufacture fills.

In [ ]:
# standard library imports
from itertools import pairwise
from math import isfinite

# third party library imports
import pandas as pd

event_results: list[dict[str, object]] = []
clocks = ["shock_received_at", "condition_available_at", "decision_at", "entry_at", "exit_at"]
outcomes = [
    "reversal_net_quote",
    "continuation_net_quote",
    "recent_return_net_quote",
    "conditional_net_quote",
]
events = pd.DataFrame(
    event_results,
    columns=["shock_id", "episode_id", "observed_state", *clocks, *outcomes, "evidence_ref"],
)

In [ ]:
if events.empty:
    print("AWAITING EVIDENCE — reversal and continuation remain competing hypotheses.")
else:
    if events.isna().any().any() or events.shock_id.duplicated().any():
        raise ValueError("Each shock requires one complete paired event record.")
    if not events.observed_state.isin(["replenished", "persistent_depletion", "timeout"]).all():
        raise ValueError("Use only states from the predeclared observed condition.")
    for field in clocks:
        if any(pd.Timestamp(value).tzinfo is None for value in events[field]):
            raise ValueError("Supply timezone-aware event timestamps.")
        events[field] = pd.to_datetime(events[field], utc=True, errors="raise")
    if events[clocks].isna().any().any():
        raise ValueError("Event timestamps cannot be NaT.")
    for earlier, later in pairwise(clocks):
        if not (events[earlier] <= events[later]).all():
            raise ValueError(f"Invalid event timing: {earlier} must precede {later}.")
    if not (events.entry_at < events.exit_at).all():
        raise ValueError("An outcome must follow entry.")
    for field in outcomes:
        events[field] = pd.to_numeric(events[field], errors="raise")
    if not events[outcomes].map(isfinite).all().all():
        raise ValueError("Paired outcomes must be finite.")
    if events.evidence_ref.astype(str).str.strip().eq("").any():
        raise ValueError("Every shock needs a retained event/trace reference.")
    events["day"] = events.decision_at.dt.strftime("%Y-%m-%d")
    events["conditional_increment_quote"] = (
        events.conditional_net_quote - events.recent_return_net_quote
    )
    display(events.groupby("observed_state")[outcomes].agg(["count", "mean"]))
    display(events.groupby("day")[["conditional_increment_quote"]].sum())
    print("Distinct shock episodes:", events.episode_id.nunique())

## 5. Falsification and economic interpretation

Inspect whether recovery happened before the condition was known or before the order could
arrive. Compare onset-based diagnostic returns with entry-based executable returns. The
former can be large while the latter is zero or negative.

Challenge the result with stale/gap exclusions, alternative predeclared depth bands,
public-trade conditioning, delay/size stresses, and episode/day concentration. Estimate
uncertainty by dependent episodes or time blocks, not by treating each update as independent.
The state means above are descriptive; they are not significance tests or a portfolio backtest.

Reject/narrow if future-defined events, gaps, a few extreme episodes or pre-entry recovery
explain the result; or if the conditional policy adds no OOS economic value over a simple
recent-return policy on the same events.

## 6. Decision and next experiment

Record the [memo](study-protocol.md#decision-memo): event counts and exclusions, how many
independent episodes/days, timing diagram, matched policy economics, concentration,
uncertainty and the inventory-aware evaluation still required.
**Current conclusion: insufficient evidence.**

If the state forecasts liquidity deterioration better than direction, test it as an
adverse-selection filter in [selective liquidity provision](liquidity-provision.ipynb).
Do not label an untradeable onset-to-recovery move a strategy.